# 04 -- Collaborative Filtering

Two user-item collaborative filtering recommenders evaluated with **precision@10**.

### Algorithms

**4a -- People similar to you listen** (user-based CF):
1. Build a sparse user x song matrix from play counts.
2. Compute cosine similarity between users.
3. For a target user, find the k most similar neighbours.
4. Recommend the top-10 songs the neighbours played that the target hasn't.

**4b -- People who listen to this track usually listen** (item-based CF):
1. Build a sparse song x user matrix from play counts.
2. Compute cosine similarity between songs.
3. For a target song, find the k most similar songs.
4. Recommend the top-10 similar songs ranked by similarity score.

### Evaluation
- **Train/test split**: each user's last 20% of interactions go to test.
- **Metric**: Precision@10 -- fraction of recommended tracks the user
  actually listened to in the test set. Target: **> 10%**.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1-10, by likelihood score (descending) |
| `artist` | Artist name |
| `title` | Track title |

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data.loader import MySpotifyRecommender
from src.models.collaborative_filtering import (
    build_user_item_matrix,
    evaluate_item_cf,
    evaluate_user_cf,
    recommend_tracks_df,
    recommend_users_df,
    train_test_split,
)


In [ ]:
rs = MySpotifyRecommender.from_files()

---
## Research

### Train / Test Split

In [ ]:
train, test = train_test_split(rs, test_ratio=0.2)
train.head()

In [ ]:
user_item, user_idx, song_idx, idx_song = build_user_item_matrix(train)

### 4a -- People similar to you listen (user-based CF)

In [ ]:
sample_user = (
    train.groupby("user_id")["play_count"].count()
    .sort_values(ascending=False)
    .index[0]
)
print(f"Sample user: {sample_user}")

recs = recommend_users_df(sample_user, user_item, user_idx, idx_song, rs.tracks)
recs

#### Precision@10 -- User-based CF

In [ ]:
avg_pk = evaluate_user_cf(rs, train, test, user_item, user_idx, idx_song)

### 4b -- People who listen to this track usually listen (item-based CF)

In [ ]:
top_song = (
    train.groupby("song_id")["play_count"].sum()
    .sort_values(ascending=False)
    .index[0]
)

seed_info = rs.tracks.drop_duplicates("song_id").set_index("song_id").loc[top_song]
print(f"Seed track: {seed_info['artist']} -- {seed_info['title']}")

recs_track = recommend_tracks_df(top_song, user_item, song_idx, idx_song, rs.tracks)
recs_track

#### Precision@10 -- Item-based CF

In [ ]:
avg_pk_item = evaluate_item_cf(rs, train, test, user_item, song_idx, idx_song)

---
## Summary

In [ ]:
print(f"{'Method':<30s} {'Precision@10':>12s}  {'Status':>6s}")
print("-" * 52)
print(f"{'4a -- User-based CF':<30s} {avg_pk:>11.4f}   {'PASS' if avg_pk > 0.10 else 'FAIL':>6s}")
print(f"{'4b -- Item-based CF':<30s} {avg_pk_item:>11.4f}   {'PASS' if avg_pk_item > 0.10 else 'FAIL':>6s}")